# Axon break detection with FAST U-Net++ — N4

Basic results only: a per-image metrics table (saved as CSV) and comparison plots (saved as PNG). No inline image display.

Data: 3-channel acquisition (AF647 tubulin, AF488, DAPI), 70.6 nm pixels, resampled to 87.7 nm. Channel 0 (AF647) is tubulin. **Only bmi and scr were acquired (no Ni).**

**Pipeline:**
1. Max-Z projection, resampled to 87.7 nm pixels.
2. Pretrained **FAST U-Net++** (BioImage.IO F-actin filament model) axon mask.
3. Skeletonize.
4. Break-pair detection by endpoint direction + gap clarity, with the same parameters as `results/`.
5. Raw-image gap-contrast check of each break (*confirmed* = dark gap).

The FAST mask replaces the Frangi + Otsu mask used in `results/`. See `results_with_Fast-Unet/README.md` and
`pretrained_axon_detection.ipynb` for why.

In [1]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pylibCZIrw import czi as pyczi
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.stats import kruskal
from itertools import combinations
from skimage.morphology import skeletonize, disk, dilation
from skimage.filters import gaussian
from skimage.measure import label
from skimage.transform import resize

DEVICE    = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR = '/Users/ctuna/Desktop/axon_codes/pretrained_models/fast_actin'

In [2]:
# ── I/O helpers: max-Z projection, resampled to a common 87.7 nm pixel size ──
TARGET_PX_UM = 0.08768704530423278    # N1 pixel size; break parameters below are in pixels at this scale

def _pixel_size_um(path):
    with pyczi.open_czi(path) as cz:
        md = cz.raw_metadata
    return float(re.findall(r'<Distance Id="X">\s*<Value>([^<]+)</Value>', md)[0]) * 1e6

def _load(path, channel=0):
    # Max-Z projection over all Z planes at T=0, channel 0 (tubulin), resampled to TARGET_PX_UM.
    with pyczi.open_czi(path) as cz:
        n_z = cz.total_bounding_box['Z'][1]
        planes = [np.squeeze(cz.read(plane={'C': channel, 'Z': z, 'T': 0}))
                  for z in range(n_z)]
    img = np.stack(planes).max(axis=0).astype(np.float32)
    scale = _pixel_size_um(path) / TARGET_PX_UM
    if abs(scale - 1) > 0.02:
        shape = tuple(int(round(s * scale)) for s in img.shape)
        img = resize(img, shape, order=1, anti_aliasing=scale < 1, preserve_range=True).astype(np.float32)
    return img

def _n_zplanes(path):
    with pyczi.open_czi(path) as cz:
        return cz.total_bounding_box['Z'][1]

In [3]:
# ── FAST U-Net++ / ResNet34 (BioImage.IO 'reliable-waterbuffalo', F-actin filament model) ──
# Filament classes: 1 = actin & stress fibers, 4 = filopodia. Their summed softmax = axon probability.
# The model min-max normalizes its input internally, so the raw projection is passed in.
sys.path.insert(0, MODEL_DIR)
from model import build_model
_fast = build_model()
_fast.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'fast_model.pth'), map_location='cpu'))
_fast = _fast.to(DEVICE).eval()
FAST_FILAMENT_CLASSES = [1, 4]

def detect_axons_fast(img, thr=0.5):
    H, W = img.shape
    x = np.pad(img.astype(np.float32), ((0, -H % 32), (0, -W % 32)), mode='reflect')
    with torch.no_grad():
        logits = _fast(torch.from_numpy(x)[None, None].to(DEVICE)).float().cpu()
    prob = torch.softmax(logits, 1)[0, FAST_FILAMENT_CLASSES].sum(0).numpy()[:H, :W]
    return prob > thr, prob

In [4]:
# ── Skeleton helpers ───────────────────────────────────────────────────────────

def find_skeleton_endpoints(skeleton):
    kernel = np.ones((3, 3), dtype=np.uint8)
    kernel[1, 1] = 0
    nc = ndimage.convolve(skeleton.astype(np.uint8), kernel, mode='constant', cval=0)
    return np.argwhere((skeleton > 0) & (nc == 1))

def filter_small_components(skeleton, min_size=15):
    labeled = label(skeleton, connectivity=2)
    sizes   = np.bincount(labeled.ravel())
    keep    = np.where(sizes >= min_size)[0]
    keep    = keep[keep != 0]
    return np.isin(labeled, keep)

def _make_skeleton(img):
    binary, prob = detect_axons_fast(img)
    return skeletonize(binary), binary, prob

In [5]:
# ── Skeleton metrics ───────────────────────────────────────────────────────────

def compute_metrics(skeleton, min_component_size=15, gap_radius=30):
    # skel_length      : total skeleton pixels
    # n_components     : connected fragments after removing tiny spurs
    # frag_rate        : n_components / skel_length x 1000
    # continuity       : skel_length / n_components  (mean fragment length px)
    # short_frac       : fraction of fragments shorter than 50 px
    # endpoint_rate    : n_endpoints / skel_length x 1000
    # gap_bridges      : number of components that merge when skeleton is dilated by gap_radius
    # gap_bridge_rate  : gap_bridges / skel_length x 1000
    skel_clean  = filter_small_components(skeleton, min_size=min_component_size).astype(bool)
    lbl         = label(skel_clean, connectivity=2)
    n_comp      = int(lbl.max())
    skel_length = int(skel_clean.sum())
    if n_comp == 0 or skel_length == 0:
        return dict(skel_length=0, n_components=0, frag_rate=0., continuity=0.,
                    short_frac=0., endpoint_rate=0., gap_bridges=0, gap_bridge_rate=0.)
    sizes           = np.bincount(lbl.ravel())[1:]
    frag_rate       = n_comp / skel_length * 1000
    continuity      = skel_length / n_comp
    short_frac      = float((sizes < 50).sum() / n_comp)
    endpoints       = find_skeleton_endpoints(skel_clean)
    endpoint_rate   = len(endpoints) / skel_length * 1000
    n_after         = int(label(dilation(skel_clean, disk(gap_radius)), connectivity=2).max())
    gap_bridges     = max(0, n_comp - n_after)
    gap_bridge_rate = gap_bridges / skel_length * 1000
    return dict(skel_length=skel_length, n_components=n_comp, frag_rate=frag_rate,
                continuity=continuity, short_frac=short_frac,
                endpoint_rate=endpoint_rate, gap_bridges=gap_bridges,
                gap_bridge_rate=gap_bridge_rate)

In [6]:
# ── Break detection — direction + gap clarity filters (unchanged from results/) ─

def _compute_tangent(skel_pts, ep, window=15):
    dists = np.linalg.norm(skel_pts - ep, axis=1)
    nearby = skel_pts[dists <= window]
    if len(nearby) < 3:
        return None
    centroid = nearby.mean(axis=0)
    vec = np.array(ep, dtype=float) - centroid
    n = np.linalg.norm(vec)
    return vec / n if n > 1e-8 else None

def _gap_is_clear(binary_mask, pt1, pt2, n_samples=30, border_frac=0.15, max_overlap=0.10):
    rs = np.linspace(pt1[0], pt2[0], n_samples)
    cs = np.linspace(pt1[1], pt2[1], n_samples)
    skip = max(1, int(n_samples * border_frac))
    rs = np.round(rs[skip:-skip]).astype(int)
    cs = np.round(cs[skip:-skip]).astype(int)
    if len(rs) == 0:
        return True
    rs = np.clip(rs, 0, binary_mask.shape[0] - 1)
    cs = np.clip(cs, 0, binary_mask.shape[1] - 1)
    return float(binary_mask[rs, cs].mean()) <= max_overlap

def find_broken_lines(skeleton, binary_mask=None, min_gap=10, max_gap=80,
                      min_component_size=15, angle_threshold=40,
                      max_gap_overlap=0.10, tangent_window=15):
    # Find axon break pairs using two filters:
    #   1. Direction: both endpoint tangents must be aligned (same axon)
    #   2. Gap clarity: the path between them must be free of mask (no crossing axon)
    skeleton   = filter_small_components(skeleton, min_size=min_component_size).astype(bool)
    endpoints  = find_skeleton_endpoints(skeleton)
    if len(endpoints) < 2:
        return [], endpoints
    skel_pts   = np.argwhere(skeleton)
    comp_map   = label(skeleton, connectivity=2)
    ep_labels  = np.array([comp_map[r, c] for r, c in endpoints])
    cos_thresh = np.cos(np.radians(angle_threshold))
    tree       = cKDTree(endpoints)
    candidates = []
    for i, j in tree.query_pairs(r=max_gap):
        if ep_labels[i] == ep_labels[j]:
            continue
        dist = float(np.linalg.norm(endpoints[i] - endpoints[j]))
        if dist < min_gap:
            continue
        gap_vec = (endpoints[j].astype(float) - endpoints[i]) / dist
        ti = _compute_tangent(skel_pts, endpoints[i], window=tangent_window)
        tj = _compute_tangent(skel_pts, endpoints[j], window=tangent_window)
        if ti is not None and tj is not None:
            if np.dot(ti, gap_vec) < cos_thresh:   continue
            if np.dot(-tj, gap_vec) < cos_thresh:  continue
            if abs(np.dot(ti, tj)) < cos_thresh:   continue
        if binary_mask is not None:
            if not _gap_is_clear(binary_mask, tuple(endpoints[i]), tuple(endpoints[j]),
                                 max_overlap=max_gap_overlap):
                continue
        candidates.append((dist, i, j))
    candidates.sort()
    used = set(); breaks = []
    for dist, i, j in candidates:
        if i not in used and j not in used:
            used.update([i, j])
            breaks.append({'pt1': tuple(endpoints[i]),
                           'pt2': tuple(endpoints[j]),
                           'distance': dist})
    return breaks, endpoints


# ── Raw-image check of each break: is the gap actually dark? ──────────────────
# gap_contrast = (gap intensity - background) / (axon intensity at the two ends - background)
# confirmed break: gap_contrast < CONFIRM_THR. Otherwise it is a mask dropout over a visible axon.
CONFIRM_THR = 0.35

def gap_contrast(img_s, bg, skel, pt1, pt2, end_radius=8):
    p1, p2 = np.array(pt1, float), np.array(pt2, float)
    t  = np.linspace(0.2, 0.8, 25)
    rr = np.clip(np.round(p1[0] + t * (p2[0] - p1[0])).astype(int), 0, img_s.shape[0] - 1)
    cc = np.clip(np.round(p1[1] + t * (p2[1] - p1[1])).astype(int), 0, img_s.shape[1] - 1)
    gap = np.median(img_s[rr, cc])
    ends = []
    for p in (p1, p2):
        r0, c0 = int(p[0]), int(p[1])
        sl = (slice(max(r0 - end_radius, 0), r0 + end_radius + 1),
              slice(max(c0 - end_radius, 0), c0 + end_radius + 1))
        v = img_s[sl][skel[sl]]
        ends.append(np.median(v) if v.size else img_s[r0, c0])
    return float((gap - bg) / (min(ends) - bg + 1e-8))

def confirm_breaks(img, binary, skel, breaks):
    img_s = gaussian(img.astype(float), 1, preserve_range=True)
    outside = ~ndimage.binary_dilation(binary, iterations=3)
    bg = np.median(img_s[outside]) if outside.any() else 0.
    return [gap_contrast(img_s, bg, skel, b['pt1'], b['pt2']) < CONFIRM_THR for b in breaks]

In [7]:
# ── Per-image analysis, CSV table, comparison plots (no image display) ───────

def analyze_directory(directory, min_component_size=15, gap_radius=30,
                      min_gap=10, max_gap=80, angle_threshold=40,
                      max_gap_overlap=0.10, tangent_window=15):
    files = sorted(f for f in os.listdir(directory) if f.endswith('.czi'))
    rows = []
    for fname in files:
        fpath = os.path.join(directory, fname)
        img = _load(fpath)
        skel, binary, _ = _make_skeleton(img)
        m = compute_metrics(skel, min_component_size, gap_radius)
        skel_clean = filter_small_components(skel, min_size=min_component_size).astype(bool)
        breaks, eps = find_broken_lines(
            skel, binary_mask=binary, min_gap=min_gap, max_gap=max_gap,
            min_component_size=min_component_size, angle_threshold=angle_threshold,
            max_gap_overlap=max_gap_overlap, tangent_window=tangent_window)
        n_conf = int(sum(confirm_breaks(img, binary, skel_clean, breaks)))
        L = max(m['skel_length'], 1)
        row = {'filename': fname, 'n_zplanes': _n_zplanes(fpath),
               'pixel_size_nm': round(_pixel_size_um(fpath) * 1000, 1),
               'n_breaks': len(breaks), 'n_endpoints': len(eps),
               'n_confirmed': n_conf,
               'break_rate': len(breaks) / L * 1000,
               'confirmed_rate': n_conf / L * 1000,
               'pct_confirmed': 100 * n_conf / max(len(breaks), 1),
               'mask_frac': float(binary.mean())}
        row.update(m)
        rows.append(row)
    return rows


def analyze_all(directories, **kwargs):
    all_rows = []
    for cond, directory in directories.items():
        for row in analyze_directory(directory, **kwargs):
            all_rows.append({'condition': cond, **row})
    return pd.DataFrame(all_rows)


def _strip_panels(df, panels, save_path, suptitle):
    labels = list(dict.fromkeys(df['condition']))
    x_pos  = np.arange(len(labels))
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    for ax, (key, ylabel, title) in zip(np.atleast_1d(axes), panels):
        groups = [df.loc[df['condition'] == l, key].to_numpy() for l in labels]
        for xi, (color, g) in enumerate(zip(colors, groups)):
            jitter = np.random.default_rng(0).uniform(-0.12, 0.12, size=len(g))
            ax.scatter(xi + jitter, g, s=70, color=color, zorder=3, alpha=0.85)
            mean = g.mean()
            sem  = g.std(ddof=1) / np.sqrt(len(g)) if len(g) > 1 else 0
            ax.plot([xi - .25, xi + .25], [mean, mean], color=color, lw=2.5, zorder=4)
            ax.errorbar(xi, mean, yerr=sem, fmt='none', color=color, capsize=5, lw=1.8, zorder=4)
        if all(len(g) >= 2 for g in groups):
            _, p_kw = kruskal(*groups)
            ax.text(0.98, 0.98, f'KW p={p_kw:.3f}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=8, style='italic',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
            y_max = max(g.max() for g in groups)
            y_r   = y_max * 0.08; y_b = y_max * 1.05
            for i_, j_ in combinations(range(len(labels)), 2):
                if len(groups[i_]) >= 2 and len(groups[j_]) >= 2:
                    _, p2 = kruskal(groups[i_], groups[j_])
                    sig = '***' if p2 < 0.001 else '**' if p2 < 0.01 else '*' if p2 < 0.05 else 'ns'
                    xm  = (x_pos[i_] + x_pos[j_]) / 2
                    yl  = y_b + (abs(i_ - j_) - 1) * y_r
                    ax.plot([x_pos[i_], x_pos[i_], x_pos[j_], x_pos[j_]],
                            [yl, yl + y_r * .4, yl + y_r * .4, yl], lw=1.2, color='black')
                    ax.text(xm, yl + y_r * .5, sig, ha='center', va='bottom', fontsize=10)
        ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=12)
        ax.set_ylabel(ylabel, fontsize=10); ax.set_title(title, fontsize=11)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved plot to {save_path}")


def plot_comparison(df, save_path, gap_radius=30, min_component_size=15):
    # 4-panel strip chart: frag_rate, endpoint_rate, gap_bridge_rate, continuity (same as results/).
    panels = [
        ('frag_rate',       'Frags / 1k skel px',     'Frag rate  (more = more breaks)'),
        ('endpoint_rate',   'Endpoints / 1k skel px', 'Endpoint rate  (more = more breaks)'),
        ('gap_bridge_rate', f'Gap bridges / 1k px\n(dilation {gap_radius}px)',
                            'Gap-bridge rate  (more = more closeable breaks)'),
        ('continuity',      'Mean segment length (px)', 'Continuity  (more = healthier)'),
    ]
    _strip_panels(df, panels, save_path,
                  f"Axon integrity (FAST U-Net++ mask) — min_size={min_component_size}, gap_radius={gap_radius}px")


def plot_breaks(df, save_path):
    # 3-panel strip chart of the detected break pairs.
    panels = [
        ('break_rate',     'Breaks / 1k skel px',           'Break rate  (all detected pairs)'),
        ('confirmed_rate', 'Confirmed breaks / 1k skel px', 'Confirmed break rate  (dark gap in raw)'),
        ('pct_confirmed',  '% of breaks confirmed',         'Share of breaks with a dark gap'),
    ]
    _strip_panels(df, panels, save_path,
                  f"Axon breaks (FAST U-Net++ mask) — confirmed: gap contrast < {CONFIRM_THR}")

## Directories and run

In [8]:
directories = {
    "bmi": "/Users/ctuna/Desktop/axon_codes/N4/bmi1 n=4/bmi1 n4_data/",
    "scr": "/Users/ctuna/Desktop/axon_codes/N4/SCR n=4_data/",
}

RESULTS_DIR = '/Users/ctuna/Desktop/axon_codes/results_with_Fast-Unet/N4'
CSV_PATH    = os.path.join(RESULTS_DIR, "csv", "axon_break_metrics_FAST_N4.csv")
PLOT_PATH   = os.path.join(RESULTS_DIR, "figures", "axon_break_comparison_FAST_N4.png")
BREAK_PATH  = os.path.join(RESULTS_DIR, "figures", "axon_break_counts_FAST_N4.png")

df = analyze_all(directories, gap_radius=30)
df.to_csv(CSV_PATH, index=False)
print(f"Saved metrics for {len(df)} images to {CSV_PATH}")

plot_comparison(df, PLOT_PATH, gap_radius=30)
plot_breaks(df, BREAK_PATH)
df

Saved metrics for 17 images to /Users/ctuna/Desktop/axon_codes/results_with_Fast-Unet/N4/csv/axon_break_metrics_FAST_N4.csv
Saved plot to /Users/ctuna/Desktop/axon_codes/results_with_Fast-Unet/N4/figures/axon_break_comparison_FAST_N4.png


Saved plot to /Users/ctuna/Desktop/axon_codes/results_with_Fast-Unet/N4/figures/axon_break_counts_FAST_N4.png


,condition,filename,n_zplanes,pixel_size_nm,n_breaks,n_endpoints,n_confirmed,break_rate,confirmed_rate,pct_confirmed,mask_frac,skel_length,n_components,frag_rate,continuity,short_frac,endpoint_rate,gap_bridges,gap_bridge_rate
0,bmi,New-01.czi,2,70.6,86,636,10,4.140189,0.481417,11.627907,0.077567,20772,148,7.124976,140.351351,0.472973,30.618140,137,6.595417
1,bmi,New-02.czi,2,70.6,50,535,9,2.307444,0.415340,18.000000,0.072436,21669,115,5.307121,188.426087,0.521739,24.689649,105,4.845632
2,bmi,New-03.czi,2,70.6,78,810,7,2.523129,0.226435,8.974359,0.113766,30914,181,5.854952,170.795580,0.513812,26.201721,179,5.790257
3,bmi,New-04.czi,1,70.6,52,527,8,3.011874,0.463365,15.384615,0.054684,17265,108,6.255430,159.861111,0.444444,30.524182,101,5.849986
4,bmi,New-05.czi,1,70.6,54,511,3,3.289674,0.182760,5.555556,0.051919,16415,109,6.640268,150.596330,0.431193,31.130064,105,6.396588
5,bmi,New-06.czi,1,70.6,98,728,9,4.628974,0.425110,9.183673,0.069793,21171,127,5.998772,166.700787,0.480315,34.386661,123,5.809834
6,bmi,New-07.czi,1,70.6,168,1302,17,4.662393,0.471790,10.119048,0.117762,36033,278,7.715150,129.615108,0.543165,36.133544,275,7.631893
7,bmi,New-08.czi,1,70.6,27,247,5,2.705140,0.500952,18.518519,0.034592,9981,59,5.911231,169.169492,0.525424,24.747019,56,5.610660
8,bmi,New-09.czi,1,70.6,46,447,4,2.851122,0.247924,8.695652,0.054686,16134,80,4.958473,201.675000,0.500000,27.705467,77,4.772530
9,scr,New-01.czi,1,70.6,24,276,1,1.824263,0.076011,4.166667,0.049250,13156,46,3.496503,286.000000,0.521739,20.979021,44,3.344482
